In [ ]:
import logging
import sys
from pathlib import Path

import cooler
import numpy as np
import pandas as pd
import torch

sys.path.insert(1, "..")

import os

from torch import nn
from torch.utils.data import DataLoader

from config.eda import DataConfig
from config.model_conf import FeaturesConfig, MLPConfig
from RNADNA_background.models.models import MLPNoiseModel
from RNADNA_background.utils.learn import (
    TabularDataset,
    eval_epoch,
    train_epoch,
    train_val_test_split,
)

In [ ]:
# configuration of the dataset
data_conf_file = Path("../config/eda_conf.ini")
params = DataConfig(data_conf_file)

# configuration of the features
features_conf_file = Path("../config/data_conf.ini")
features_params = FeaturesConfig(features_conf_file, params.chromosomes)

model_params = MLPConfig("../config/model_conf.ini", "MLPNoiseModel")

# cool file to get bins coordinates
cool_file = params.hic_folder / Path(
    f"{params.cell_line}.mcool::resolutions/{params.bin_size}"
)
c = cooler.Cooler(str(cool_file))
sample_path = f"_{params.sample}" if params.sample else ""

In [ ]:
logdir = Path("../logs")
logging.basicConfig(
    level=logging.INFO,
    filename=logdir / "mlp_model.log",
    filemode="a",
    format="%(asctime)s %(levelname)s %(message)s",
)

In [ ]:
# data loading
windows = pd.read_csv(
    params.learn_data_folder
    / f"windows_{features_params.window_size}_{features_params.shift}.tsv",
    sep="\t",
)

features = pd.read_csv(
    params.learn_data_folder / f"features_{params.bin_size}_refactor.tsv",
    sep="\t",
    usecols=[
        "chrom",
        "start",
        "end",
        "bin",
        *features_params.num_features,
        *features_params.cat_features,
    ],
)

if params.sample:
    contacts = pd.read_csv(
        params.data_path
        / f"binned_contacts_{params.bin_size}_{params.sample}.tsv",
        sep="\t",
    )
else:
    contacts = pd.read_csv(
        params.data_path / f"binned_contacts_{params.bin_size}.tsv",
        sep="\t",
    )

print(contacts["count"].mean())
perc_99 = np.percentile(contacts["count"], features_params.perc_mask)
contacts.loc[contacts["count"] > perc_99, "count"] = perc_99

In [ ]:
# train val test split
splitted_windows, splitted_features, splitted_contacts = train_val_test_split(
    windows,
    features,
    contacts,
    features_params.train_chromosomes,
    features_params.val_chromosomes,
    features_params.test_chromosomes,
)
train_windows, val_windows, test_windows = splitted_windows
train_features, val_features, test_features = splitted_features
train_contacts, val_contacts, test_contacts = splitted_contacts

In [ ]:
# datasets initialization
# avg. features calculation is inside dataset
win_size = features_params.window_size // params.bin_size
win_type = None

train_ds = TabularDataset(
    windows=train_windows,
    features=train_features,
    contacts=train_contacts,
    features_params=features_params,
    mean_features=True,
    win_size=win_size,
    win_type=win_type,
)
val_ds = TabularDataset(
    windows=val_windows,
    features=val_features,
    contacts=val_contacts,
    features_params=features_params,
    mean_features=True,
    win_size=win_size,
    win_type=win_type,
    scaler=train_ds.scaler,
)
test_ds = TabularDataset(
    windows=test_windows,
    features=test_features,
    contacts=test_contacts,
    features_params=features_params,
    mean_features=True,
    win_size=win_size,
    win_type=win_type,
    scaler=train_ds.scaler,
)

In [ ]:
train_loader = DataLoader(
    train_ds, shuffle=True, batch_size=model_params.batch_size
)
val_loader = DataLoader(val_ds, shuffle=False, batch_size=512)
test_loader = DataLoader(test_ds, shuffle=False, batch_size=512)

In [ ]:
device = "cuda"
activations = {"SELU": nn.SELU, "ReLU": nn.ReLU}

model = MLPNoiseModel(
    train_ds[0][0].shape[0],
    activation=activations[model_params.activation_func],
    hidden=model_params.hidden,
)
pytorch_total_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
model = model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=model_params.lr,
    weight_decay=model_params.weight_decay,
)
max_lr = model_params.lr * model_params.div_factor

sheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lr,
    div_factor=model_params.div_factor,
    steps_per_epoch=len(train_loader),
    epochs=model_params.num_epochs,
)

criterion = nn.MSELoss()

In [ ]:
path_to_model = Path(
    f"../models/{model._get_name()}_{params.experiment}_{params.cell_line}{sample_path}{params.bin_size}"
)

In [ ]:
# training loop
val_losses = []
val_sccs = []
val_mean_scc = []
epochs_batch_loss = []
for i in range(model_params.num_epochs):
    train_loss = train_epoch(
        model,
        train_loader,
        criterion,
        device,
        optimizer,
        sheduler,
        mask_zeros=False,
    )
    val_loss, val_scc, preds, targets, epoch_loss_list = eval_epoch(
        model, val_loader, criterion, device, mask_zeros=False
    )
    print(
        f"Epoch: {i+1}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}, Val SCC: {val_scc[0]:.3f}"
    )
    epochs_batch_loss.append(epoch_loss_list)
    val_losses.append(val_loss)
    val_sccs.append(val_scc[0])


if not Path(path_to_model).is_dir():
    os.makedirs(path_to_model)
torch.save(
    model.state_dict(),
    path_to_model
    / f"v1_{model_params.hidden}_{model_params.num_epochs}_{features_params.features_group}.pth",
)
logging.info(
    f"{params}, {model_params} \n {features_params} \n  train bins number: {len(train_ds)}, best val loss: {min(val_losses)}, best val scc: {max(val_sccs)}"
)